Pandas performance depends on **three pillars:**
1. **Vectorization** (avoid Python loops)
2. **Memory layout** (dtypes, categoricals)
3. **Algorithmic choices** (groupby, merge, apply)

### Avoid Python Loops
#### Python Loop (Slow)
```python
result = []
for x in df["salary']:
    result.append(x * 1.1)
df["salary_inc"] = result
```

#### Vectorized (Fast)
```python
df["salary_inc"] = df["salary"] * 1.1
```

### `apply()` vs Vectorization
#### Very Slow
```python
df["tax"] = df["salary"].apply(lambda x: x * 0.3)
```

#### Fast
```python
df["tax"] = df["salary"] * 0.3
```

### Row-wise `apply(axis=1)` 
#### Worst-case pattern
```python
df["label"] = df.apply(
    lambda r: high if r["salary"] > 80000 else "low",
    axis=1
)
```

#### Vectorized alternative
```python
df["label"] = np.where(df["salary"] > 80000, "high", "low")
```

`axis=1` forces Python to touch **every row.**

### Memory Optimization with dtypes
#### Default (wastes memory)
```python
df.dtypes
```

#### Optimize integers
```python
df["age"] = df["age"].astype("Int16")
```

#### Optimize floats
```python
df["salary"] = df["salary"].astype("Float32")
```

#### Use nullable dtypes
```python
df["age"] = df["age"].astype("Int64")
```

### Categorical Data
#### Object dtype (slow & large)
```python
df["city"].dtype #object
```

#### Category dtype
```python
df["city"] = df["city"].astype("category")
```

- Faster groupby
- Faster comparisons
- Huge memory savings

### GroupBy Performance Rules
#### Slow
```python
df.groupby("city").apply(custom_func)
```

#### Fast
```python
df.groupby("city").agg({
    "salary": "mean",
    "age": "mean"
})
```

### Merge Performance
#### Before merging — reduce size
```python
order_small = orders[["user_id", "amount"]]
```

#### Set index when joining repeatedly
```python
users = users.set_index("user_id")
orders = orders.set_index("user_id")

users.join(orders, how="left")
```

Index joins are faster than column joins.

### Sorting & Indexing for Speed
#### Sorting helps slicing & joins
```python
df = df.sort_values("date")
```

#### Use index for frequent lookups
```python
df = df.set_index("user_id")
df.loc[102]
```

Index lookups are **O(log n)** vs scans.

### Chunking Large Files
#### Reading large CSVs
```python
chunks = pd.read_csv("data.csv", chunksize=100_000)

result = []
for chunk in chunks:
    result.append(process(chunk))

df = pd.concat(result)
```

### Avoid Copies & Chained Indexing
#### Create copies (Bad)
```python
df[df["age"] > 30]["salary"] = 0
```

#### Good
```python
df.loc[df["age"] > 30, "salary"] = 0
```

### Measure Performance
#### Time Operations
```python
%timeit df["salary"] * 1.1
```

#### Memory usage
```python
df.memory_usage(deep=True)
```

### When Pandas Is NOT Enough
If data is:
- 10–20 million rows
- multi-GB
- heavy joins / aggregations

Then consider:
- **Polars** (Rust, lazy)
- **DuckDB** (SQL engine)
- **Spark** (distributed)

### Summary Table
| Problem        | Solution            |
| -------------- | ------------------- |
| Slow loops     | Vectorization       |
| High memory    | Optimize dtypes     |
| Slow groupby   | `agg` / `transform` |
| Repeated joins | Index + join        |
| Large files    | Chunking            |
| Slow strings   | `category`          |

### DataFrame Setup

In [17]:
import pandas as pd
import numpy as np

In [18]:
np.random.seed(42)

N = 1_000_000

df = pd.DataFrame({
    "user_id": np.random.randint(1, 100_000, size=N),
    "age": np.random.randint(18, 70, size=N),
    "salary": np.random.randint(30_000, 150_000, size=N),
    "city": np.random.choice(
        ["Pune", "Mumbai", "Delhi", "Bangalore"],
        size=N
    )
})

### Exercise 1
1. Create a column `salary_inc_loop` using a Python loop.
2. Create a column `salary_inc_vec` using vectorization.
3. Measure execution time for both.
4. Explain the difference.

#### Python Loop

In [19]:
salary_inc_loop = []

for x in df["salary"]:
    salary_inc_loop.append(x * 1.1)

df["salary_inc_loop"] = salary_inc_loop

#### Vectorized

In [20]:
df["salary_inc_vec"] = df["salary"] * 1.1

- Loop -> Python runs **1,000,000 iterations**
- Vectorization -> NumPy runs **C-level SIMD operations**
- Same logic, **100×–200× faster**

### Exercise 2
1. Create column `tax_apply` using:
```python
df["salary"].apply(lambda x: x * 0.3)
```
2. Create column `tax_vec` using vectorization.
3. Time both.
4. Explain why one is slower.

#### Using `apply()`

In [21]:
df["tax_apply"] = df["salary"].apply(lambda x: x * 0.3)

#### Vectorized

In [22]:
df["tax_vec"] = df["salary"] * 0.3

- `apply()` = Python function called **1M times**
- Vectorization = NumPy ufunc in C

### Exercise 3
1. Create column `label_apply`:
    - `"high"` if salary > 80000 else `"low"`
2. Rewrite using `np.where`
3. Measure timing difference.
4. Explain what `axis=1` actually does internally.

#### Row-wise apply

In [23]:
df["label_apply"] = df.apply(
    lambda r: "high" if r["salary"] > 80000 else "low",
    axis=1
)

#### Vectorized alternative

In [24]:
df["label_vec"] = np.where(df["salary"] > 80000, "high", "low")

**`axis=1` does internally**
- Converts **each row** to a Pandas Series
- Calls Python function per row
- Massive object creation + Python overhead

**Never use `axis=1` on large data unless absolutely unavoidable.**

### Exercise 4
1. Measure memory usage **before optimization.**
2. Convert:
   - `city` -> `category`
   - `age` -> `Int16`
3. Measure memory usage **after optimization.**
4. Compute percentage reduction.

#### Memory BEFORE

In [25]:
mem_before = df.memory_usage(deep=True).sum() / (1024**2)
mem_before

np.float64(206.15411281585693)

#### Optimize dtypes

In [26]:
df["city"] = df["city"].astype("category")
df["age"] = df["age"].astype("Int16")

#### Memory AFTER

In [27]:
mem_after = df.memory_usage(deep=True).sum() / (1024**2)
mem_after

np.float64(149.8867702484131)

#### % Reduction

In [28]:
((mem_before - mem_after) / mem_before) * 100

np.float64(27.293824895797027)

**Explanation**
- `object` strings -> heavy Python objects
- `category` -> integer codes + lookup table
- Smaller memory -> faster CPU + fewer cache misses

### Exercise 5
1. Compute average salary per city using:
   - `groupby().apply()`
   - `groupby().agg()`
2. Time both.
2. Explain why one is faster.

#### Using `apply`

In [29]:
def avg_salary(group):
    return group["salary"].mean()

df.groupby("city").apply(avg_salary)

/var/folders/6t/gr1hsb2x571bqm8sycqy4h440000gn/T/ipykernel_5136/780878524.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("city").apply(avg_salary)
/var/folders/6t/gr1hsb2x571bqm8sycqy4h440000gn/T/ipykernel_5136/780878524.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("city").apply(avg_salary)


city
Bangalore    90047.601517
Delhi        89977.742893
Mumbai       90071.672727
Pune         90101.260391
dtype: float64

#### Using `agg`

In [30]:
df.groupby("city").agg(
    avg_salary=("salary", "mean")
    )

/var/folders/6t/gr1hsb2x571bqm8sycqy4h440000gn/T/ipykernel_5136/854136500.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("city").agg(


,avg_salary
city,
Bangalore,90047.601517
Delhi,89977.742893
Mumbai,90071.672727
Pune,90101.260391


`agg()` is faster -> optimized C code

### Exercise 6
1. Create another DataFrame:
```python
orders = pd.DataFrame({
    "user_id": np.random.randint(1, 100_000, size=300_000),
    "amount": np.random.randint(100, 5000, size=300_000)
})
```
2. Merge with `df` using `merge`.
3. Set `user_id` as index in both and repeat using `join`.
4. Compare performance.
5. Explain why index joins are faster.

In [31]:
orders = pd.DataFrame({
    "user_id": np.random.randint(1, 100_000, size=300_000),
    "amount": np.random.randint(100, 5000, size=300_000)
})

#### Column-based merge

In [32]:
%timeit pd.merge(df, orders, on="user_id", how="left")

149 ms ± 2.77 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


#### Index-based join

In [34]:
df_idx = df.set_index("user_id")
order_idx = orders.set_index("user_id")

%timeit df_idx.join(order_idx, how="left")

113 ms ± 3.24 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**Index joins are faster**
- Hash lookup on index
- Less column scanning
- Better memory locality

### Exercise 7
1. Simulate reading data in chunks:
```python
for chunk in np.array_split(df, 10):
    process(chunk)
```
2. Compute total salary sum across chunks.
3. Explain why chunking prevents memory crashes.

In [35]:
total = 0

for chunk in np.array_split(df, 10):
    total += chunk["salary"].sum()

total

/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


np.int64(90049589463)

**Why chunking matters**
- Limits peak memory usage
- Prevents OOM crashes
- Enables streaming large datasets

Used in:
- CSV ingestion
- ETL pipelines
- Feature computation